# Notebook 01 — Data Extraction & Representative Sampling

**Author:** Section D – Group 8  
**Dataset:** US Real Estate Listings (Zillow)  
**Goal:** Load the raw 600k dataset, understand its structure, and extract a representative, unbiased 80k subset that preserves the original distribution of property types, states, and missing values.

## 0. Setup — Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW_PATH = '../data/raw_data.csv'
SUBSET_PATH = '../data/raw_extracted_data.csv'

os.makedirs('../reports', exist_ok=True)
print('Setup complete.')

Setup complete.


## 1. Load Raw Data

In [2]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Total rows   : {len(df):,}')
print(f'Total columns: {df.shape[1]}')

Total rows   : 600,000
Total columns: 28


In [3]:
# Preview first 5 rows
df.head()

,property_url,property_id,address,street_name,apartment,city,state,latitude,longitude,postcode,price,bedroom_number,bathroom_number,price_per_unit,living_space,land_space,land_space_unit,broker_id,property_type,property_status,year_build,total_num_units,listing_age,RunDate,agency_name,agent_name,agent_phone,is_owned_by_zillow
0,https://www.zillow.com/homedetails/3-Plat-83-1...,2064212272,"3 Plat #83-10, Wrangell, AK 99929",Plat #83-10,NaN,Wrangell,AK,NaN,NaN,99929,135000.00,NaN,NaN,NaN,NaN,3.89,acres,NaN,LOT,FOR_SALE,NaN,NaN,-1,2022-04-24 07:34:15,Anchor Properties,NaN,NaN,0
1,https://www.zillow.com/homedetails/117-3rd-St-...,249518113,"117 3rd St, Wrangell, AK 99929",3rd St,NaN,Wrangell,AK,56.47,-132.39,99929,589500.00,3.00,3.00,237.00,2478.00,7492.00,sqft,NaN,SINGLE_FAMILY,FOR_SALE,NaN,NaN,-1,2022-04-24 07:34:15,NaN,NaN,NaN,0
2,https://www.zillow.com/homedetails/LOT-2A-Fron...,2077729913,"LOT 2A Front St, Wrangell, AK 99929",LOT 2A Front St,NaN,Wrangell,AK,56.47,-132.39,99929,99999.00,NaN,0.00,NaN,NaN,7222.00,sqft,NaN,LOT,FOR_SALE,NaN,NaN,-1,2022-04-24 07:34:15,NaN,NaN,NaN,0
3,https://www.zillow.com/homedetails/LOT-A-Plat-...,2067488502,"LOT A Plat #2009-6, Wrangell, AK 99929",LOT A Plat #2009-6,NaN,Wrangell,AK,NaN,NaN,99929,495000.00,3.00,1.00,330.00,1500.00,61.97,acres,NaN,SINGLE_FAMILY,FOR_SALE,NaN,NaN,-1,2022-04-24 07:34:15,"Petersburg Properties, LLC",NaN,NaN,0
4,https://www.zillow.com/homedetails/335-Cassiar...,249518139,"335 Cassiar St, Wrangell, AK 99929",Cassiar St,NaN,Wrangell,AK,56.48,-132.39,99929,405000.00,5.00,3.00,194.00,2080.00,10436.00,sqft,NaN,SINGLE_FAMILY,FOR_SALE,NaN,NaN,-1,2022-04-24 07:34:15,NaN,NaN,NaN,0


In [4]:
# Column names & dtypes
df.dtypes

property_url           object
property_id             int64
address                object
street_name            object
apartment              object
city                   object
state                  object
latitude              float64
longitude             float64
postcode               object
price                 float64
bedroom_number        float64
bathroom_number       float64
price_per_unit        float64
living_space          float64
land_space            float64
land_space_unit        object
broker_id             float64
property_type          object
property_status        object
year_build            float64
total_num_units       float64
listing_age             int64
RunDate                object
agency_name            object
agent_name            float64
agent_phone           float64
is_owned_by_zillow      int64
dtype: object

In [5]:
# Basic info summary
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600000 entries, 0 to 599999
Data columns (total 28 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   property_url        600000 non-null  object 
 1   property_id         600000 non-null  int64  
 2   address             600000 non-null  object 
 3   street_name         599853 non-null  object 
 4   apartment           14815 non-null   object 
 5   city                599999 non-null  object 
 6   state               599999 non-null  object 
 7   latitude            529122 non-null  float64
 8   longitude           529122 non-null  float64
 9   postcode            599970 non-null  object 
 10  price               600000 non-null  float64
 11  bedroom_number      443845 non-null  float64
 12  bathroom_number     471733 non-null  float64
 13  price_per_unit      435365 non-null  float64
 14  living_space        447847 non-null  float64
 15  land_space          515119 non-nul

## 2. Understand the Raw Data

In [6]:
# Numeric summary statistics
df.describe()

,property_id,latitude,longitude,price,bedroom_number,bathroom_number,price_per_unit,living_space,land_space,broker_id,year_build,total_num_units,listing_age,agent_name,agent_phone,is_owned_by_zillow
count,600000.00,529122.00,529122.00,600000.00,443845.00,471733.00,435365.00,447847.00,515119.00,0.00,0.00,0.00,600000.00,0.00,0.00,600000.00
mean,888504216.64,36.28,-105.81,605187.04,3.41,2.47,484.04,4298.39,3099.43,NaN,NaN,NaN,-1.00,NaN,NaN,0.00
std,972470819.22,5.67,13.46,3175555.12,22.49,5.68,29855.27,178020.05,5350.08,NaN,NaN,NaN,0.00,NaN,NaN,0.02
min,27.00,18.99,-165.41,0.00,0.00,0.00,0.00,0.00,-13068.00,NaN,NaN,NaN,-1.00,NaN,NaN,0.00
25%,54021427.25,32.61,-117.35,170000.00,3.00,2.00,155.00,1360.00,1.01,NaN,NaN,NaN,-1.00,NaN,NaN,0.00
50%,206608997.50,35.40,-101.90,369900.00,3.00,2.00,220.00,1863.00,60.00,NaN,NaN,NaN,-1.00,NaN,NaN,0.00
75%,2066866866.50,39.66,-95.35,625000.00,4.00,3.00,331.00,2574.00,6534.00,NaN,NaN,NaN,-1.00,NaN,NaN,0.00
max,2146997363.00,71.30,-87.53,2147483647.00,13210.00,1892.00,15900000.00,59913270.00,1746756.00,NaN,NaN,NaN,-1.00,NaN,NaN,1.00


In [7]:
# Missing values — absolute counts
null_count = df.isnull().sum().sort_values(ascending=False)
print('Missing values per column:')
print(null_count)

Missing values per column:
agent_name            600000
total_num_units       600000
broker_id             600000
year_build            600000
agent_phone           600000
apartment             585185
price_per_unit        164635
bedroom_number        156155
agency_name           155476
living_space          152153
bathroom_number       128267
land_space             84881
land_space_unit        84881
longitude              70878
latitude               70878
street_name              147
postcode                  30
city                       1
state                      1
RunDate                    0
listing_age                0
property_url               0
property_status            0
property_type              0
property_id                0
price                      0
address                    0
is_owned_by_zillow         0
dtype: int64


In [8]:
# Missing values — percentage
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Missing value % per column:')
print(null_pct.round(2))

Missing value % per column:
agent_name           100.00
total_num_units      100.00
broker_id            100.00
year_build           100.00
agent_phone          100.00
apartment             97.53
price_per_unit        27.44
bedroom_number        26.03
agency_name           25.91
living_space          25.36
bathroom_number       21.38
land_space            14.15
land_space_unit       14.15
longitude             11.81
latitude              11.81
street_name            0.02
postcode               0.00
city                   0.00
state                  0.00
RunDate                0.00
listing_age            0.00
property_url           0.00
property_status        0.00
property_type          0.00
property_id            0.00
price                  0.00
address                0.00
is_owned_by_zillow     0.00
dtype: float64


In [9]:
# Property type distribution in full dataset
print('Property Type Distribution (Full Data):')
print(df['property_type'].value_counts())
print()
print('Share (%):')
print((df['property_type'].value_counts(normalize=True) * 100).round(2))

Property Type Distribution (Full Data):
property_type
SINGLE_FAMILY    354366
LOT              154662
CONDO             36814
TOWNHOUSE         20108
MANUFACTURED      17220
MULTI_FAMILY      16481
APARTMENT           349
Name: count, dtype: int64

Share (%):
property_type
SINGLE_FAMILY   59.06
LOT             25.78
CONDO            6.14
TOWNHOUSE        3.35
MANUFACTURED     2.87
MULTI_FAMILY     2.75
APARTMENT        0.06
Name: proportion, dtype: float64


In [10]:
# Property status distribution
print('Property Status Distribution:')
print(df['property_status'].value_counts())

Property Status Distribution:
property_status
FOR_SALE    383365
PENDING     216635
Name: count, dtype: int64


In [11]:
# Top 15 states by listing count
print('Top 15 States by Listing Count:')
print(df['state'].value_counts().head(15))

Top 15 States by Listing Count:
state
TX    146636
CA    102464
IL     43985
AZ     37494
MO     32039
WA     30460
CO     27927
LA     24387
OK     22472
OR     20557
AR     18347
NV     16585
UT     15268
ID     12862
NM     12703
Name: count, dtype: int64


In [12]:
# Price range check
df['price'] = pd.to_numeric(df['price'], errors='coerce')
print(f"Min Price : ${df['price'].min():,.0f}")
print(f"Max Price : ${df['price'].max():,.0f}")
print(f"Mean Price: ${df['price'].mean():,.0f}")
print(f"Zero-price rows: {(df['price'] == 0).sum():,}")
print(f"Null-price rows: {df['price'].isnull().sum():,}")

Min Price : $0
Max Price : $2,147,483,647
Mean Price: $605,187
Zero-price rows: 477
Null-price rows: 0


In [13]:
# Duplicate check
dupes = df.duplicated(subset='property_id').sum()
print(f'Duplicate property_id rows: {dupes:,}')

Duplicate property_id rows: 0


## 3. Representative 80k Sampling

We use `random_state=42` for reproducibility.  
Pre-filtering: Remove zero-price and placeholder postcodes before sampling so our subset is free of clearly invalid source rows.

In [14]:
# Pre-filter invalid rows before sampling
df_valid = df[(df['price'] > 0) & (df['postcode'] != '11111')]
print(f'Valid rows available for sampling: {len(df_valid):,}')

Valid rows available for sampling: 599,522


In [15]:
# Take 80k sample
SAMPLE_SIZE = 80000
df_sample = df_valid.sample(n=SAMPLE_SIZE, random_state=42)
df_sample = df_sample.reset_index(drop=True)
print(f'Sample shape: {df_sample.shape}')

Sample shape: (80000, 28)


## 4. Validate Sampling Quality — Bias Check

In [16]:
# Compare null % between raw and sample
comparison = pd.DataFrame({
    'Raw Null %':    (df_valid.isnull().sum() / len(df_valid) * 100).round(2),
    'Sample Null %': (df_sample.isnull().sum() / len(df_sample) * 100).round(2)
})
comparison['Difference'] = (comparison['Sample Null %'] - comparison['Raw Null %']).abs().round(2)
print('Null % preserved in sample:')
comparison

Null % preserved in sample:


,Raw Null %,Sample Null %,Difference
property_url,0.00,0.00,0.00
property_id,0.00,0.00,0.00
address,0.00,0.00,0.00
street_name,0.02,0.03,0.01
apartment,97.53,97.50,0.03
city,0.00,0.00,0.00
state,0.00,0.00,0.00
latitude,11.82,11.77,0.05
longitude,11.82,11.77,0.05
postcode,0.01,0.00,0.01


In [17]:
# Compare property_type distribution
raw_dist  = df_valid['property_type'].value_counts(normalize=True).rename('Raw %') * 100
samp_dist = df_sample['property_type'].value_counts(normalize=True).rename('Sample %') * 100
pd.concat([raw_dist, samp_dist], axis=1).round(2)

,Raw %,Sample %
property_type,,
SINGLE_FAMILY,59.04,59.38
LOT,25.79,25.52
CONDO,6.14,6.16
TOWNHOUSE,3.35,3.38
MANUFACTURED,2.87,2.79
MULTI_FAMILY,2.74,2.70
APARTMENT,0.06,0.07


In [18]:
# Compare state distribution (top 10)
raw_state  = df_valid['state'].value_counts(normalize=True).head(10).rename('Raw %') * 100
samp_state = df_sample['state'].value_counts(normalize=True).head(10).rename('Sample %') * 100
pd.concat([raw_state, samp_state], axis=1).round(2)

,Raw %,Sample %
state,,
TX,24.44,24.58
CA,17.08,17.09
IL,7.33,7.39
AZ,6.25,6.29
MO,5.34,5.22
WA,5.08,4.98
CO,4.65,4.57
LA,4.07,3.97
OK,3.75,3.75


In [19]:
# Price distribution comparison
print('=== Price Distribution Comparison ===')
print(f"{'':25} {'Raw':>12}  {'Sample':>12}")
for stat in ['mean', 'median', 'std']:
    rv = getattr(df_valid['price'], stat)()
    sv = getattr(df_sample['price'], stat)()
    print(f"{stat:25} ${rv:>11,.0f}  ${sv:>11,.0f}")

=== Price Distribution Comparison ===
                                   Raw        Sample
mean                      $    605,669  $    609,908
median                    $    369,900  $    370,000
std                       $  3,176,775  $  1,652,564


## 5. Save the Sample

In [20]:
df_sample.to_csv(SUBSET_PATH, index=False)
print(f'Saved {len(df_sample):,} rows to: {SUBSET_PATH}')
print(f'File size: {os.path.getsize(SUBSET_PATH) / 1024 / 1024:.1f} MB')

Saved 80,000 rows to: ../data/raw_extracted_data.csv
File size: 22.4 MB
